# 스마트 플래시카드 코치 (Smart Flashcard Coach) 설계 문서

## 1. 개요
* **이름**: 스마트 플래시카드 코치 (Smart Flashcard Coach)
* **목적**: 학습용 텍스트 파일(`learning_material.txt`)을 불러와서 한글 플래시카드를 자동 생성하고, 퀴즈 테스트를 진행한 뒤 한글 채점 보고서를 피드백합니다.

## 2. 핵심 기능
1. **학습 자료 파싱 (parse_material)**: 한글 텍스트에서 핵심 키워드 및 개념 리스트를 LLM을 통해 정제 및 추출합니다.
2. **플래시카드 생성 (generate_flashcards)**: 추출된 개념 기반으로 한글 Q&A 쌍 세트를 구성합니다.
3. **대화형 퀴즈 출제 (quiz_interrupt)**: 사용자에게 한글 질문 세트를 전달하고 답변을 제출할 때까지 대기(Interrupt)합니다.
4. **퀴즈 채점 및 피드백 (grade_quiz)**: 사용자의 답변과 정답을 매칭해 한글로 작성된 결과 리포트를 산출합니다.

## 3. 그래프 구조
```
[START] -> parse_material -> generate_flashcards -> quiz_interrupt -> grade_quiz -> [END]
```

In [ ]:
from typing import TypedDict, List, Dict, Any
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt
from langchain.chat_models import init_chat_model

# 1. State 정의
class State(TypedDict):
    learning_material: str
    concepts: List[str]
    flashcards: List[Dict[str, str]]
    user_answers: List[str]
    evaluation_report: str

# LLM 초기화 (한글 처리를 위해 OpenAI gpt-4o-mini 모델 활용)
llm = init_chat_model("openai:gpt-4o-mini")

In [ ]:
# 2. 노드 정의
def parse_material(state: State) -> Dict[str, Any]:
    print("--- [노드] parse_material 실행 ---")
    material = state.get("learning_material", "")
    
    # 한글 프롬프트 설계
    prompt = f"""
    제공된 학습 자료에서 핵심적인 교육 개념이나 전문 용어를 추출해 주세요.
    각 개념은 불필요한 설명이나 번호, 불릿 기호 없이 한 줄에 하나씩 명확한 단어나 핵심 어구로만 추출해 주세요.
    
    학습 자료:
    {material}
    """
    
    response = llm.invoke(prompt)
    concepts = [line.strip() for line in response.content.split("\n") if line.strip()]
    return {"concepts": concepts}

def generate_flashcards(state: State) -> Dict[str, Any]:
    print("--- [노드] generate_flashcards 실행 ---")
    concepts = state.get("concepts", [])
    flashcards = []
    
    for i, concept in enumerate(concepts):
        # 3단계 고도화: 각 핵심 개념에 대한 쉬운 정의를 LLM을 사용하여 한국어로 동적 생성
        prompt = f"""
        학습용 플래시카드를 작성하고 있습니다. 다음 핵심 개념에 대한 쉽고 명확한 정의와 설명을 한국어로 작성해 주세요.
        답변은 1~2문장의 명확한 한국어 설명으로만 작성해 주세요.
        
        핵심 개념:
        {concept}
        """
        response = llm.invoke(prompt)
        
        flashcards.append({
            "id": i + 1,
            "question": f"'{concept}'의 개념과 정의는 무엇인가요?",
            "answer": response.content.strip()
        })
    return {"flashcards": flashcards}

def quiz_interrupt(state: State) -> Dict[str, Any]:
    print("--- [노드] quiz_interrupt 실행 (사용자 피드백 대기) ---")
    flashcards = state.get("flashcards", [])
    # 사용자 입력을 받기 위해 interrupt 호출 (한글 안내)
    answer = interrupt({
        "questions": [card["question"] for card in flashcards],
        "instruction": "'user_answers' 키값에 각 질문에 대한 답변 목록을 담아 전송해 주세요."
    })
    return {
        "user_answers": answer.get("user_answers", [])
    }

def grade_quiz(state: State) -> Dict[str, Any]:
    print("--- [노드] grade_quiz 실행 ---")
    flashcards = state.get("flashcards", [])
    user_answers = state.get("user_answers", [])
    
    report = "### 퀴즈 평가 결과 보고서\n"
    for i, card in enumerate(flashcards):
        u_ans = user_answers[i] if i < len(user_answers) else "답변 없음"
        report += f"\n질문 {card['id']}: {card['question']}\n- 제출한 답변: {u_ans}\n- 모범 답안: {card['answer']}\n"
    
    return {"evaluation_report": report}

In [ ]:
# 3. 그래프 연결
graph_builder = StateGraph(State)

graph_builder.add_node("parse_material", parse_material)
graph_builder.add_node("generate_flashcards", generate_flashcards)
graph_builder.add_node("quiz_interrupt", quiz_interrupt)
graph_builder.add_node("grade_quiz", grade_quiz)

graph_builder.add_edge(START, "parse_material")
graph_builder.add_edge("parse_material", "generate_flashcards")
graph_builder.add_edge("generate_flashcards", "quiz_interrupt")
graph_builder.add_edge("quiz_interrupt", "grade_quiz")
graph_builder.add_edge("grade_quiz", END)

# 메모리 세이버와 함께 컴파일
from langgraph.checkpoint.memory import InMemorySaver
memory = InMemorySaver()
graph = graph_builder.compile(checkpointer=memory)

In [11]:
# 2단계 단위 테스트: parse_material 노드 개별 검증 (learning_material.txt 로드)
import os

file_path = "learning_material.txt"
if os.path.exists(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        material_data = f.read()
else:
    material_data = "지도학습은 정답이 있는 데이터를 학습합니다.\n비지도학습은 정답이 없는 데이터를 학습합니다."

mock_state_1 = {
    "learning_material": material_data
}
print("--- [단위 테스트] parse_material 노드 단독 실행 ---")
output_1 = parse_material(mock_state_1)

print("\n=== [2단계 단위 검증] 결과 ===")
concepts = output_1.get("concepts", [])
for c in concepts:
    print(f"- {c}")

assert len(concepts) > 0, "오류: 핵심 개념이 추출되지 않았습니다."
print("\n2단계 parse_material 단위 테스트 통과!")

--- [단위 테스트] parse_material 노드 단독 실행 ---
--- [노드] parse_material 실행 ---

=== [2단계 단위 검증] 결과 ===
- 머신러닝
- 지도학습
- 비지도학습
- 강화학습
- 분류
- 회귀
- 스팸 메일 필터링
- 집값 예측
- 정답
- 데이터셋
- 알고리즘
- 군집화
- 차원 축소
- 세그먼트 분석
- 이상치 탐지
- 에이전트
- 환경
- 보상
- 정책
- 시행착오
- 자율주행 자동차
- 로봇 제어
- 게임 AI

2단계 parse_material 단위 테스트 통과!


In [ ]:
# 3단계 단위 테스트: generate_flashcards 노드 개별 검증
mock_state_2 = {
    "concepts": concepts
}
print("--- [단위 테스트] generate_flashcards 노드 단독 실행 ---")
output_2 = generate_flashcards(mock_state_2)

print("\n=== [3단계 단위 검증] 결과 ===")
flashcards = output_2.get("flashcards", [])
for card in flashcards:
    print(f"\n[카드 {card['id']}]")
    print(f"질문: {card['question']}")
    print(f"답변: {card['answer']}")

assert len(flashcards) == len(mock_state_2["concepts"]), "오류: 입력 개념 수와 플래시카드 수가 불일치합니다."
print("\n3단계 generate_flashcards 단위 테스트 통과!")

--- [단위 테스트] generate_flashcards 노드 단독 실행 ---
--- [노드] generate_flashcards 실행 ---

=== [3단계 단위 검증] 결과 ===

[카드 1]
질문: '머신러닝'의 개념과 정의는 무엇인가요?
답변: '머신러닝'에 대한 상세 설명 및 정의 예시입니다.

[카드 2]
질문: '지도학습'의 개념과 정의는 무엇인가요?
답변: '지도학습'에 대한 상세 설명 및 정의 예시입니다.

[카드 3]
질문: '비지도학습'의 개념과 정의는 무엇인가요?
답변: '비지도학습'에 대한 상세 설명 및 정의 예시입니다.

[카드 4]
질문: '강화학습'의 개념과 정의는 무엇인가요?
답변: '강화학습'에 대한 상세 설명 및 정의 예시입니다.

[카드 5]
질문: '분류'의 개념과 정의는 무엇인가요?
답변: '분류'에 대한 상세 설명 및 정의 예시입니다.

[카드 6]
질문: '회귀'의 개념과 정의는 무엇인가요?
답변: '회귀'에 대한 상세 설명 및 정의 예시입니다.

[카드 7]
질문: '스팸 메일 필터링'의 개념과 정의는 무엇인가요?
답변: '스팸 메일 필터링'에 대한 상세 설명 및 정의 예시입니다.

[카드 8]
질문: '집값 예측'의 개념과 정의는 무엇인가요?
답변: '집값 예측'에 대한 상세 설명 및 정의 예시입니다.

[카드 9]
질문: '정답'의 개념과 정의는 무엇인가요?
답변: '정답'에 대한 상세 설명 및 정의 예시입니다.

[카드 10]
질문: '데이터셋'의 개념과 정의는 무엇인가요?
답변: '데이터셋'에 대한 상세 설명 및 정의 예시입니다.

[카드 11]
질문: '알고리즘'의 개념과 정의는 무엇인가요?
답변: '알고리즘'에 대한 상세 설명 및 정의 예시입니다.

[카드 12]
질문: '군집화'의 개념과 정의는 무엇인가요?
답변: '군집화'에 대한 상세 설명 및 정의 예시입니다.

[카드 13]
질문: '차원 축소'의 개념과 정의는 무엇인가요?
답변: '차원 축소'에 대한 상세 설명 및 정의 예시입니다.

[카드

In [ ]:
# 4. 통합 실행: 텍스트 파일 로드 및 첫 단계 실행
import os
config = {"configurable": {"thread_id": "notebook_session_korean"}}

file_path = "learning_material.txt"
if os.path.exists(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        material_data = f.read()
else:
    material_data = "비디오 분석 기반 핵심 개념 요약본 (예시)"

initial_state = {"learning_material": material_data}
print("--- [통합 테스트] 그래프 최초 실행 시작 ---")
result = graph.invoke(initial_state, config=config)
print("\n인터럽트 중단 상태 도달. (사용자 퀴즈 답변 대기 중)")